# 175. QK-Norm 与 Attention Logit Softcapping：为什么能稳定注意力？

> **面试问题：Q/K 归一化、可学习温度和 logit softcap 分别解决什么，怎样手写并验证不会破坏 mask？**

## 先给结论

标准点积注意力只用 `1/sqrt(d)` 控制初始化尺度，训练中 Q/K 范数仍可能增长并让 softmax 饱和。QK-Norm 把方向和范数解耦，再用可学习 scale 控制温度；softcapping 用有界平滑函数限制极端 logit。二者都是稳定性工具，不保证质量提升，也不能代替稳定 softmax、正确 mask 与训练监控。

## 推荐回答主线

1. 先诊断 logit 幅度、attention entropy 和饱和比例，区分输入尺度与方向信息。
2. 沿 head_dim 对 Q/K 做 L2 或 RMS 归一化，使用正值可学习 scale，明确各轴形状。
3. 实现 `cap*tanh(logit/cap)`，检查有界、零点斜率和 mask 顺序。
4. 比较梯度、熵和极端输入；上线绑定 head 数、norm epsilon、scale/cap 与 kernel 版本。

## 教学实现边界

这里实现一个小型 causal attention 模块；不复现 Gemma 2 完整架构，也不声称 QK-Norm 与 softcapping 必须同时使用。实际收益依赖初始化、优化器、dtype、序列长度和融合 kernel。

## 一手资料

- [Query-Key Normalization for Transformers](https://arxiv.org/abs/2010.04245)
- [Gemma 2 Technical Report](https://arxiv.org/abs/2408.00118)
- [Scaling Vision Transformers to 22B](https://arxiv.org/abs/2302.05442)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass  # 导入本单元需要的依赖。

import warnings  # 导入本单元需要的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import torch  # 导入本单元需要的依赖。
from torch import nn  # 导入本单元需要的依赖。

# 小张量同时覆盖多头、causal mask 和不同输入幅度。
torch.manual_seed(175)  # 计算并保存当前步骤的中间状态。
B, H, L, D = 2, 3, 6, 8  # 计算并保存当前步骤的中间状态。
q = torch.randn(B, H, L, D)  # 计算并保存当前步骤的中间状态。
k = torch.randn(B, H, L, D)  # 计算并保存当前步骤的中间状态。
v = torch.randn(B, H, L, D)  # 计算并保存当前步骤的中间状态。

assert q.shape == k.shape == v.shape  # 用受控断言验证关键不变量。
assert D % 2 == 0  # 用受控断言验证关键不变量。
assert torch.isfinite(q).all()  # 用受控断言验证关键不变量。


## 1. 先量化饱和：logit 大不等于信息更强

输入整体放大时，标准点积 logit 按范数乘积放大，softmax 趋近 one-hot，非最大位置梯度变小。应同时看最大绝对值、熵和最大概率，而不是只看 loss 是否有限。


In [ ]:
def stable_softmax(logits, dim=-1, valid_mask=None):  # 定义本节可复用的核心函数。
    # 显式维护有效集合；全遮罩行的指数和为零，约定返回全零而不是 NaN。
    finite = torch.isfinite(logits)  # 计算并保存当前步骤的中间状态。
    if valid_mask is None:  # 按当前条件选择后续控制路径。
        valid_mask = finite  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        valid_mask = torch.broadcast_to(valid_mask.to(torch.bool), logits.shape) & finite  # 计算并保存当前步骤的中间状态。
    masked_logits = logits.masked_fill(~valid_mask, -torch.inf)  # 计算并保存当前步骤的中间状态。
    row_max = masked_logits.amax(dim=dim, keepdim=True)  # 计算并保存当前步骤的中间状态。
    safe_max = torch.where(torch.isfinite(row_max), row_max, torch.zeros_like(row_max))  # 计算并保存当前步骤的中间状态。
    exponential = torch.exp(masked_logits - safe_max).masked_fill(~valid_mask, 0.0)  # 计算并保存当前步骤的中间状态。
    denominator = exponential.sum(dim=dim, keepdim=True)  # 计算并保存当前步骤的中间状态。
    return exponential / denominator.clamp_min(torch.finfo(exponential.dtype).tiny)  # 返回当前分支计算出的结果。

def entropy(probabilities):  # 定义本节可复用的核心函数。
    return -(probabilities * probabilities.clamp_min(1e-12).log()).sum(-1)  # 返回当前分支计算出的结果。

# 把 Q/K 同时放大十倍，logit 放大百倍且平均注意力熵下降。
base_logits = q @ k.transpose(-1, -2) / math.sqrt(D)  # 计算并保存当前步骤的中间状态。
large_logits = (10 * q) @ (10 * k).transpose(-1, -2) / math.sqrt(D)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(large_logits, 100 * base_logits, atol=1e-4)  # 用受控断言验证关键不变量。
assert entropy(stable_softmax(large_logits)).mean() < entropy(stable_softmax(base_logits)).mean()  # 用受控断言验证关键不变量。
assert stable_softmax(base_logits).sum(-1).allclose(torch.ones(B, H, L))  # 用受控断言验证关键不变量。

# 反例：普通的 -inf 减 -inf 会产生 NaN；安全实现把全遮罩行定义为零分布。
all_masked_logits = torch.full((2, 4), -torch.inf)  # 计算并保存当前步骤的中间状态。
all_masked_probability = stable_softmax(all_masked_logits)  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(all_masked_probability).all()  # 用受控断言验证关键不变量。
assert torch.equal(all_masked_probability, torch.zeros_like(all_masked_probability))  # 用受控断言验证关键不变量。


## 2. QK-Norm：只保留方向，再用正值 scale 控温

沿每个 head 的最后一维归一化，使点积落在有限范围。learnable scale 常通过 exp/softplus 保证为正；它替代或组合 `1/sqrt(d)` 的细节要以具体模型配置为准，不能重复缩放而不自知。


In [ ]:
def l2_normalize_heads(tensor, eps=1e-6):  # 定义本节可复用的核心函数。
    return tensor / tensor.square().sum(-1, keepdim=True).clamp_min(eps).sqrt()  # 返回当前分支计算出的结果。

def qk_norm_logits(query, key, log_scale):  # 定义本节可复用的核心函数。
    scale = log_scale.exp()[None, :, None, None]  # 计算并保存当前步骤的中间状态。
    return (l2_normalize_heads(query) @ l2_normalize_heads(key).transpose(-1, -2)) * scale  # 返回当前分支计算出的结果。

# 归一化后每个非零 head 向量范数为 1，logit 绝对值不超过对应 scale。
log_scale = torch.full((H,), math.log(math.sqrt(D)), requires_grad=True)  # 计算并保存当前步骤的中间状态。
norm_logits = qk_norm_logits(q, k, log_scale)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(l2_normalize_heads(q).norm(dim=-1), torch.ones(B, H, L), atol=1e-5)  # 用受控断言验证关键不变量。
assert norm_logits.abs().amax() <= log_scale.exp().max() + 1e-5  # 用受控断言验证关键不变量。
assert norm_logits.shape == (B, H, L, L)  # 用受控断言验证关键不变量。


## 3. 尺度不变性：Q/K 范数变化不再改变注意力

只要缩放因子为正，L2 QK-Norm 对 Q/K 的整体或逐 token 幅度缩放不敏感。这能抑制范数漂移，但也意味着模型不能再用 Q/K 范数直接编码置信，表达能力改由方向与 scale 承担。


In [ ]:
# 给每个 token 不同正缩放，QK-Norm logits 应保持不变；标准点积不会。
q_scale = torch.linspace(0.2, 3.0, L)[None, None, :, None]  # 计算并保存当前步骤的中间状态。
k_scale = torch.linspace(2.5, 0.3, L)[None, None, :, None]  # 计算并保存当前步骤的中间状态。
scaled_norm_logits = qk_norm_logits(q * q_scale, k * k_scale, log_scale)  # 计算并保存当前步骤的中间状态。
scaled_standard = (q * q_scale) @ (k * k_scale).transpose(-1, -2) / math.sqrt(D)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(norm_logits, scaled_norm_logits, atol=1e-5)  # 用受控断言验证关键不变量。
assert not torch.allclose(base_logits, scaled_standard)  # 用受控断言验证关键不变量。
assert (q_scale > 0).all() and (k_scale > 0).all()  # 用受控断言验证关键不变量。


## 4. Softcapping：平滑限制极值，而不是硬 clip

`c*tanh(x/c)` 的输出在 `[-c,c]`，零附近近似恒等，极端值梯度平滑趋零。硬 clip 在边界不可导且边界外梯度为零；softcap 仍需选择合适 c，过小会把所有注意力压平。


In [ ]:
def softcap(logits, cap):  # 定义本节可复用的核心函数。
    if cap <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("cap 必须为正")  # 遇到非法合同立即显式失败。
    return cap * torch.tanh(logits / cap)  # 返回当前分支计算出的结果。

# 输出有界；小 logit 近似不变；奇函数保持正负对称。
probe = torch.tensor([-100.0, -0.01, 0.0, 0.01, 100.0], requires_grad=True)  # 计算并保存当前步骤的中间状态。
capped = softcap(probe, 5.0)  # 计算并保存当前步骤的中间状态。
assert capped.abs().max() <= 5.0  # 用受控断言验证关键不变量。
assert torch.allclose(capped[1:4], probe[1:4], atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(capped, -softcap(-probe, 5.0))  # 用受控断言验证关键不变量。


## 5. 完整 causal attention：cap 后统一执行 query/key mask

不能对 `-inf` mask 值做 softcap，否则会把非法位置重新变成有限数。这里先 cap 有效 logits，再构造 causal、padding key 与 padding query 的交集；masked softmax 对全空行返回零，因此 padding query 和无可用 key 的行都保持有限零输出。


In [ ]:
class StableAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, heads, cap=6.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.log_scale = nn.Parameter(torch.full((heads,), math.log(math.sqrt(D))))  # 计算并保存当前步骤的中间状态。
        self.cap = cap  # 计算并保存当前步骤的中间状态。

    def forward(self, query, key, value, valid_key=None, valid_query=None):  # 定义本节可复用的核心函数。
        if query.ndim != 4 or key.shape != value.shape or query.shape[:2] != key.shape[:2]:  # 按当前条件选择后续控制路径。
            raise ValueError("Q/K/V 必须是兼容的 [batch, head, length, dim]")  # 遇到非法合同立即显式失败。
        logits = softcap(qk_norm_logits(query, key, self.log_scale), self.cap)  # 计算并保存当前步骤的中间状态。
        batch, _, query_length, _ = query.shape  # 计算并保存当前步骤的中间状态。
        key_length = key.shape[-2]  # 计算并保存当前步骤的中间状态。
        query_position = torch.arange(query_length, device=query.device)[:, None]  # 计算并保存当前步骤的中间状态。
        key_position = torch.arange(key_length, device=query.device)[None, :]  # 计算并保存当前步骤的中间状态。
        allowed = (key_position <= query_position)[None, None].expand(batch, 1, -1, -1)  # 计算并保存当前步骤的中间状态。
        if valid_key is not None:  # 按当前条件选择后续控制路径。
            if valid_key.shape != (batch, key_length):  # 按当前条件选择后续控制路径。
                raise ValueError("valid_key 形状必须是 [batch, key_length]")  # 遇到非法合同立即显式失败。
            allowed = allowed & valid_key[:, None, None, :]  # 计算并保存当前步骤的中间状态。
        if valid_query is not None:  # 按当前条件选择后续控制路径。
            if valid_query.shape != (batch, query_length):  # 按当前条件选择后续控制路径。
                raise ValueError("valid_query 形状必须是 [batch, query_length]")  # 遇到非法合同立即显式失败。
            allowed = allowed & valid_query[:, None, :, None]  # 计算并保存当前步骤的中间状态。
        probability = stable_softmax(logits, valid_mask=allowed)  # 计算并保存当前步骤的中间状态。
        output = probability @ value  # 计算并保存当前步骤的中间状态。
        return output, probability  # 返回当前分支计算出的结果。

# 未来概率严格为零、每个有候选的行归一化，输出形状与 query 轴一致。
module = StableAttention(H)  # 计算并保存当前步骤的中间状态。
output, probability = module(q, k, v)  # 计算并保存当前步骤的中间状态。
causal = torch.ones(L, L, dtype=torch.bool).tril()[None, None]  # 计算并保存当前步骤的中间状态。
assert output.shape == (B, H, L, D)  # 用受控断言验证关键不变量。
assert probability.masked_select(~causal).abs().sum() == 0  # 用受控断言验证关键不变量。
assert torch.allclose(probability.sum(-1), torch.ones(B, H, L), atol=1e-6)  # 用受控断言验证关键不变量。

# 正数逐 token 缩放必须经真实 forward 仍保持概率与输出不变，而不只比较中间 logits。
scaled_output, scaled_probability = module(q * q_scale, k * k_scale, v)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(scaled_probability, probability, atol=1e-5)  # 用受控断言验证关键不变量。
assert torch.allclose(scaled_output, output, atol=1e-5)  # 用受控断言验证关键不变量。

# padding query/key 与“整条样本没有有效 key”都进入主路径，非法概率为零且没有 NaN。
valid_key = torch.tensor([[1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 0]], dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
valid_query = torch.tensor([[1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 0]], dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
masked_output, masked_probability = module(q, k, v, valid_key, valid_query)  # 计算并保存当前步骤的中间状态。
invalid_keys = (~valid_key[:, None, None, :]).expand_as(masked_probability)  # 计算并保存当前步骤的中间状态。
invalid_queries = (~valid_query[:, None, :, None]).expand_as(masked_probability)  # 计算并保存当前步骤的中间状态。
assert masked_probability.masked_select(invalid_keys).abs().sum() == 0  # 用受控断言验证关键不变量。
assert masked_probability.masked_select(invalid_queries).abs().sum() == 0  # 用受控断言验证关键不变量。
assert masked_output.masked_select((~valid_query[:, None, :, None]).expand_as(masked_output)).abs().sum() == 0  # 用受控断言验证关键不变量。

empty_first_key = valid_key.clone(); empty_first_key[0] = False  # 计算并保存当前步骤的中间状态。
empty_output, empty_probability = module(q, k, v, empty_first_key, valid_query)  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(empty_probability).all() and torch.isfinite(empty_output).all()  # 用受控断言验证关键不变量。
assert empty_probability[0].abs().sum() == 0 and empty_output[0].abs().sum() == 0  # 用受控断言验证关键不变量。


## 6. 梯度诊断：极端输入下仍要检查 scale 与 Q/K 梯度

归一化和 softcap 都可能在极端区间减小梯度。训练监控应按层/头记录 Q/K norm、logit p99、entropy、scale、梯度范数与 NaN，而不能只证明 forward 有限。


In [ ]:
# 使用放大输入反传，检查参数与输入梯度均有限且非零。
q_grad = (q * 100).detach().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
k_grad = (k * 100).detach().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
out_grad, _ = module(q_grad, k_grad, v)  # 计算并保存当前步骤的中间状态。
loss = out_grad.square().mean()  # 计算并保存当前步骤的中间状态。
loss.backward()  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(q_grad.grad).all() and q_grad.grad.norm() > 0  # 用受控断言验证关键不变量。
assert torch.isfinite(k_grad.grad).all() and k_grad.grad.norm() > 0  # 用受控断言验证关键不变量。
assert module.log_scale.grad is not None and torch.isfinite(module.log_scale.grad).all()  # 用受控断言验证关键不变量。


## 7. 熵与温度校准：不同 head 不应机械共享一个阈值

scale 越大通常越尖锐，但每个 head 的内容分布不同。可在校准集上约束 entropy 分位或异常饱和率，并保留无需低熵的复制/归纳头差异；这不是让所有 head 熵相同。


In [ ]:
def attention_report(logits):  # 定义本节可复用的核心函数。
    probs = stable_softmax(logits)  # 计算并保存当前步骤的中间状态。
    return {  # 返回当前分支计算出的结果。
        "mean_entropy": float(entropy(probs).mean()),  # 执行当前语句以推进本节示例。
        "mean_max_prob": float(probs.max(-1).values.mean()),  # 执行当前语句以推进本节示例。
        "saturated_fraction": float((probs.max(-1).values > 0.99).float().mean()),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

# 更高全局 scale 产生更低熵和更高最大概率，报告值在合法范围。
low = attention_report(qk_norm_logits(q, k, torch.full((H,), math.log(1.0))))  # 计算并保存当前步骤的中间状态。
high = attention_report(qk_norm_logits(q, k, torch.full((H,), math.log(20.0))))  # 计算并保存当前步骤的中间状态。
assert high["mean_entropy"] < low["mean_entropy"]  # 用受控断言验证关键不变量。
assert high["mean_max_prob"] > low["mean_max_prob"]  # 用受控断言验证关键不变量。
assert 0 <= high["saturated_fraction"] <= 1  # 用受控断言验证关键不变量。


## 8. 制品合同：norm、scale 与 cap 属于 checkpoint 语义

若服务 kernel 忽略 QK-Norm 或使用错误 cap，权重仍能加载但输出分布会静默改变。配置摘要要绑定 norm 类型/epsilon、scale 参数化、cap、head_dim、dtype 和 kernel；发布比较 loss、长上下文、熵、吞吐与溢出率。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class AttentionConfig:  # 定义承载本节状态与行为的数据结构。
    head_dim: int  # 执行当前语句以推进本节示例。
    heads: int  # 执行当前语句以推进本节示例。
    qk_norm: str  # 执行当前语句以推进本节示例。
    epsilon: float  # 执行当前语句以推进本节示例。
    scale_parameterization: str  # 执行当前语句以推进本节示例。
    softcap: float  # 执行当前语句以推进本节示例。

def config_hash(config):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(config), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# 任一数学配置变化都必须改变摘要，线上维度与模块一致。
config = AttentionConfig(D, H, "l2", 1e-6, "exp", 6.0)  # 计算并保存当前步骤的中间状态。
digest = config_hash(config)  # 计算并保存当前步骤的中间状态。
assert len(digest) == 64  # 用受控断言验证关键不变量。
assert config.heads == module.log_scale.numel()  # 用受控断言验证关键不变量。
assert digest != config_hash(AttentionConfig(D, H, "l2", 1e-6, "exp", 30.0))  # 用受控断言验证关键不变量。


## 面试收束：从公式走到生产合同

建议用六步回答：目标与约束、张量/数据合同、核心公式、正确性反例、质量—成本评测、版本与回滚。Notebook 的小模型只证明机制和边界，不代表论文规模结果、真实 GPU kernel 加速或线上泛化。生产替换时仍应保留同一批 oracle，并补齐目标硬件 profiling、分布式一致性、数据 provenance、安全审计和灰度发布。

继续追问时要主动区分：训练期方法与已有 checkpoint 的后处理、理论 FLOPs 与 wall-clock、平均质量与关键 slice、可逆近似与不可逆状态、模型置信与校准后的决策概率。
